# 01. PySpark Introduction & Architecture

Welcome to the **PySpark Step-by-Step Tutorial Series**!

This notebook is your starting point. You will learn:
1. **What is Apache Spark & PySpark?**
2. **Why use PySpark instead of Pandas?**
3. **The Spark Architecture (Driver, Workers, Executors, Cluster Manager)**
4. **Core Execution Concepts (Lazy Evaluation, DAG, Transformations vs Actions)**
5. **How to initialize a SparkSession with proper configurations**
6. **How to safely stop your SparkSession to release memory and RAM**


---
## 1. What is Apache Spark and PySpark?

* **Apache Spark** is an open-source, distributed computing engine designed to process huge volumes of data (terabytes or petabytes) quickly by distributing the workload across multiple computers (called a cluster).
* **PySpark** is the Python API for Apache Spark. It lets you write familiar Python code while Spark runs heavy data processing across distributed machines behind the scenes using its fast Java Virtual Machine (JVM) core.

### 🍳 Simple Analogy: Pandas vs PySpark
* **Pandas = A Single Chef in a Kitchen.**
  * If the chef has to prepare 5 meals, they do it very quickly.
  * If 5,000 customers arrive at once, the chef is overwhelmed and crashes because their hands and counter space (RAM) are limited.
* **PySpark = A Head Chef managing 50 Line Cooks.**
  * The head chef (Driver) breaks the 5,000 orders into small batches and sends them to 50 cooks (Executors).
  * All cooks work simultaneously in parallel. The entire dinner is prepared in record time!


---
## 2. When Should You Use PySpark vs Pandas?

| Feature | Pandas | PySpark |
| :--- | :--- | :--- |
| **Data Size** | Small to Medium (Fits in computer RAM, e.g. < 5-10 GB) | Big Data (10s of GB, Terabytes, Petabytes) |
| **Architecture** | Single-node (runs on 1 CPU / 1 machine) | Distributed (runs on multiple CPUs or cluster of machines) |
| **Execution** | Eager (executes immediately line by line) | Lazy (plans optimization first, executes on Action) |
| **Fault Tolerance** | None (if RAM runs out, your program crashes) | High (if a machine fails, Spark automatically recalculates) |
| **Speed** | Extremely fast on small data | Faster on large distributed datasets |


---
## 3. Spark Cluster Architecture

When you run PySpark, several components work together:

1. **Driver Program**:
   * The "Brain" or "Boss".
   * Runs your main Python script and creates the `SparkSession`.
   * Breaks your code into tasks and coordinates execution across the cluster.
2. **Cluster Manager**:
   * The "Resource Allocator".
   * Manages hardware resources (RAM, CPU cores). Examples: Spark Standalone, YARN, Kubernetes, Mesos, or `local` mode on your laptop.
3. **Worker Nodes**:
   * The physical or virtual computers in the cluster that provide CPU and RAM.
4. **Executors**:
   * The actual worker processes running on Worker Nodes.
   * They store cached data in memory and execute individual tasks assigned by the Driver.


---
## 4. Key Spark Concepts You Must Know

### A. In-Memory Computing
Traditional big data tools (like Hadoop MapReduce) write intermediate results to hard disks between steps. Spark keeps intermediate results directly in **RAM**, making it up to **100x faster** than Hadoop!

### B. Lazy Evaluation
Spark does **NOT** compute results immediately when you write transformations.
* Instead, it takes notes: *"User wants to filter column A, then group by column B, then calculate the average."*
* It builds an optimized execution plan called a **DAG (Directed Acyclic Graph)**.
* Spark only runs the computation when an **Action** is called (like showing rows, saving to disk, or counting).

### C. Transformations vs Actions
* **Transformations (Lazy):** Do not calculate immediately. They return a new DataFrame.
  * Examples: `filter()`, `select()`, `groupBy()`, `withColumn()`, `join()`.
* **Actions (Eager):** Trigger the actual computation and return results to the Driver or write to storage.
  * Examples: `show()`, `count()`, `collect()`, `first()`, `write.save()`.

```
Raw Data ---> Transformation (Lazy) ---> Transformation (Lazy) ---> Action (Eager: Computation Runs!)
```


---
## 5. Initializing Your First SparkSession

The `SparkSession` is the entry point to programming Spark with the Dataset and DataFrame API.
Let's create one now!

On Windows, Python 3.12 is the recommended kernel for this tutorial. Set the worker interpreter before starting Spark; if you change kernels, restart the notebook first.

* **`appName("...")`**: Names your application (shows up in monitoring dashboards).
* **`master("local[*]")`**:
  * `local`: Runs Spark locally on your machine instead of a distributed cluster.
  * `[*]`: Tells Spark to use all available CPU cores on your computer.
* **`getOrCreate()`**: Creates a new session, or reuses an existing one if already open.


In [1]:


from pyspark.sql import SparkSession

# Step 1: Build the SparkSession
spark = SparkSession.builder \
    .appName("01_PySpark_Introduction") \
    .master("local[*]") \
    .getOrCreate()

# Step 2: Set log level to WARN to reduce unnecessary INFO messages in your notebook
spark.sparkContext.setLogLevel("WARN")

print("SparkSession successfully created!")
print(f"Spark Version: {spark.version}")
print(f"Application Name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/05 17:57:36 WARN Utils: Your hostname, Tiger, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/05 17:57:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 17:57:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession successfully created!
Spark Version: 4.2.0
Application Name: 01_PySpark_Introduction
Master: local[*]


---
## 6. Accessing the Spark Web UI

Spark includes a built-in web monitoring interface called the **Spark UI**.
While your SparkSession is active, you can open your web browser and navigate to:

👉 **`http://localhost:4040`**

In the Spark UI, you can see:
* Active and completed Jobs and Stages
* DAG visualization of your queries
* Memory and storage usage
* Executors and thread activity


---
## 7. Creating Your First Simple DataFrame

Let's create a tiny DataFrame to see Spark in action.


In [2]:
# Sample data: list of tuples (Name, Department, Salary)
data = [
    ("Alice", "Engineering", 85000),
    ("Bob", "Marketing", 62000),
    ("Charlie", "Engineering", 92000),
    ("Diana", "HR", 58000),
    ("Evan", "Marketing", 67000)
]

# Column names
columns = ["Name", "Department", "Salary"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Action: display the DataFrame table
print("Employee DataFrame:")
df.head(5)


Employee DataFrame:


[Row(Name='Alice', Department='Engineering', Salary=85000),
 Row(Name='Bob', Department='Marketing', Salary=62000),
 Row(Name='Charlie', Department='Engineering', Salary=92000),
 Row(Name='Diana', Department='HR', Salary=58000),
 Row(Name='Evan', Department='Marketing', Salary=67000)]

In [3]:
# Check the schema (data types of columns)
print("DataFrame Schema:")
df.printSchema()


DataFrame Schema:
root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: long (nullable = true)



In [4]:
# Transformation (Lazy) + Action (Eager)
# Notice: df.filter() is a transformation; .show() is the action that runs it
high_earners = df.filter(df.Salary > 65000)

print("Employees earning more than $65,000:")
high_earners.show()


Employees earning more than $65,000:
+-------+-----------+------+
|   Name| Department|Salary|
+-------+-----------+------+
|  Alice|Engineering| 85000|
|Charlie|Engineering| 92000|
|   Evan|  Marketing| 67000|
+-------+-----------+------+



---
## 8. Summary of Notebook 01
* PySpark is the Python library for Apache Spark distributed computing.
* It uses a **Driver-Executor** architecture coordinated by a **Cluster Manager**.
* It processes data in **RAM** and uses **Lazy Evaluation** to optimize queries before running.
* **Transformations** build the recipe; **Actions** trigger execution.
* The `SparkSession` is the main entry point to PySpark.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession

### Why is this essential?
* A running `SparkSession` holds an active Java Virtual Machine (JVM) process with worker threads and allocated memory.
* If you leave notebooks running without stopping Spark, multiple sessions will accumulate in the background, consuming your computer's RAM and CPU.
* **Always run `spark.stop()` at the end of every notebook.**


In [5]:
# STOP THE SPARK SESSION TO FREE SYSTEM RAM AND RESOURCES
spark.stop()
print("SparkSession stopped successfully! RAM and background JVM resources have been released.")


SparkSession stopped successfully! RAM and background JVM resources have been released.
